In [142]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [143]:
import os
import sys
import subprocess
import pandas as pd
from strip_ansi import strip_ansi
from IPython.display import display
from lxml import etree
from conlanger.tools.SoundChangeRule import SoundChangeRule

RULES_DIR = "./data/rules"
OVERWRITE_RULES = True

In [144]:
if OVERWRITE_RULES:
    tree = etree.parse("./data/index_diachronica.xml")
    root = tree.getroot()

    results = []
    rules = []

    for child in root:
        index = child.attrib["index"]
        results.append({
            "index": index,
            "name": child.attrib["name"],
            "rule_count": len(child.findall("rule")),
        })

        rules.append((index, str(SoundChangeRule(child, format="asca"))))


    sections_df = pd.DataFrame(results)

    sections_df.to_csv("./data/index_diachronica_sections.csv", index=False)
    
    for index, rule in rules:
        with open(f"./data/rules/asca/{index}.rsca", "w") as f:
            f.write(rule)

    sections_df.head(3)

In [145]:
sections_df = pd.read_csv("./data/index_diachronica_sections.csv", dtype={"index": str, "name": str, "rule_count": int})

rules_df = sections_df[sections_df["rule_count"] > 0].copy()
rules_df['asca_rule_file'] = rules_df['index'].apply(lambda x: f"{x}.rsca")
rules_df['brassica_rule_file'] = rules_df['index'].apply(lambda x: f"{x}.bsc")

display(rules_df.head(3))

asca_rule_files = rules_df['asca_rule_file'].tolist()
brassica_rule_files = rules_df['brassica_rule_file'].tolist()


,index,name,rule_count,asca_rule_file,brassica_rule_file
1,6.1,Proto-Afro-Asiatic to Proto-Omotic,12,6.1.rsca,6.1.bsc
2,6.1.1,Proto-Omotic to North Omotic,18,6.1.1.rsca,6.1.1.bsc
3,6.1.1.1,North Omotic to Bench,9,6.1.1.1.rsca,6.1.1.1.bsc


In [146]:
# run asca-rust to validate rules

asca_results = []
asca_word_file = "./data/words/asca/weirdness_0.5.wsca"
asca_alias_file = "./data/asca_aliases.alias"

def run_asca(asca_word_file, rule_file):
    cmd = f"~/.cargo/bin/asca run {asca_word_file} --rules {RULES_DIR}/asca/{rule_file}"# --alias {asca_alias_file}"

    result = {"rule": rule_file, "returncode": 0, "error": ""}

    try:
        output = subprocess.run(cmd, capture_output=True, timeout=10, shell=True, text=True)
        output.check_returncode()

    except subprocess.CalledProcessError as exc:
        result["returncode"] = exc.returncode 
        result["error"] = strip_ansi(exc.stderr.strip()).replace('\n', ' ')
    except subprocess.TimeoutExpired as exc:
        result["returncode"] = 124
        result["error"] = exc.output.decode("utf-8").replace('\n', ' ')

    return result

for rule_file in asca_rule_files[:100]:
    result = run_asca(asca_word_file, rule_file)
    asca_results.append(result)

asca_results_df = pd.DataFrame(asca_results)

asca_results_df.to_csv("./data/asca_results.csv", index=False)

print(asca_results_df["returncode"].value_counts())

errors = asca_results_df[asca_results_df["returncode"] != 0][["rule", "error"]]

for r in errors.itertuples(index=False):
    print(r.rule, r.error)

returncode
1    75
0    25
Name: count, dtype: int64
6.2.2.1.2.rsca Syntax Error: Expected number, but received ɡ      |     CVʕ > ħʔ // C = ɡw     |                     ^     @ Rule 3, Line 4
6.2.2.1.3.rsca Syntax Error: Can't have segments after the end of a word     |     r > l / #(C)_c(C)# ?     |                      ^     @ Rule 2, Line 7
6.2.2.1.4.rsca Syntax Error: Options can only be used in Environments or Structures     |     ɣ(w) > ɡ / #_Vx     |      ^^^     @ Rule 3, Line 2
6.2.2.1.5.rsca Runtime Error: A Set in output must have a matching Set in input     |     s > {ʃ,h}     |         ^^^^^     @ Rule 2, Line 5
6.2.2.1.6.rsca Syntax Error: Unknown grouping 'R'. Known groupings are (C)onsonant, (O)bstruent, (S)onorant, (P)losive, (F)ricative, (L)iquid, (N)asal, (G)lide, and (V)owel     |     S > [+ voice] / {V,R}_V     |                        ^     @ Rule 2, Line 3
6.2.2.1.7.rsca Syntax Error: Expected '_', but received ''     |     h > ∅ / in coda     |                 

In [147]:
# run Brassica to validate rules

# brassica_results = []
# brassica_word_file = "./data/words/brassica/weirdness_0.5.lex"

# def run_brassica(brassica_word_file, rule_file):
#     cmd = f"brassica {RULES_DIR}/brassica/{rule_file} -i {brassica_word_file}"

#     result = {"rule": rule_file, "returncode": 0, "error": ""}

#     try:
#         print(cmd)
#         output = subprocess.run(cmd, capture_output=True, timeout=10, shell=True, text=True)
#         print(output.stdout)
#         output.check_returncode()

#     except subprocess.CalledProcessError as exc:
#         result["returncode"] = exc.returncode 
#         result["error"] = exc.output.replace("\n", "\\n")
#     except subprocess.TimeoutExpired as exc:
#         result["returncode"] = 124
#         result["error"] = exc.output.decode("utf-8").replace("\n", "\\n")

#     return result

# for rule_file in brassica_rule_files[:10]:
#     result = run_brassica(brassica_word_file, rule_file)
#     brassica_results.append(result)

# brassica_results_df = pd.DataFrame(brassica_results)

# brassica_results_df.to_csv("./data/brassica_results.csv", index=False)

# brassica_results_df[brassica_results_df["returncode"] != 0].head()